# Data Extraction for Google Colab

This notebook is set up for Colab:
- mount Google Drive
- stream the ViMD dataset from Hugging Face
- resample audio to 24 kHz
- use VAD to trim leading/trailing silence from training clips
- create 15s-ish segment candidates from long clips while preserving silence/inter-utterance gaps
- optionally transcribe long clips via PhoWhisper
- keep unaligned long-clip segments out of the training manifest unless configured otherwise
- write files to fast local Colab storage first, then sync to Drive

## 1) Install dependencies

After this cell finishes in Colab, restart the runtime. Then rerun the notebook from the imports/config cell onward.


In [10]:
#%pip uninstall -y torch torchaudio torchvision
#%pip install torch==2.8.0 torchaudio==2.8.0
#%pip uninstall -y torchvision

In [11]:
#%pip install datasets soundfile pandas librosa huggingface_hub transformers accelerate silero-vad

## 2) Mount Google Drive

Extraction itself will write to local Colab storage first for better throughput.

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## 3) Hugging Face login

In [ ]:
# from huggingface_hub import login
# login("") #replace with ur huggingface token

In [14]:
# from huggingface_hub import whoami
# whoami()

## 4) Imports and config

In [15]:
import csv
import gc
import io
import json
import os
import re
import shutil
from pathlib import Path
import warnings

os.environ['DISABLE_SAFETENSORS_CONVERSION'] = '1'

import librosa
import numpy as np
import pandas as pd
import soundfile as sf
import torch
from datasets import load_dataset
from silero_vad import get_speech_timestamps, load_silero_vad
from transformers import WhisperForConditionalGeneration, WhisperProcessor, logging as hf_logging

hf_logging.set_verbosity_error()
warnings.filterwarnings("ignore")

In [16]:
DATASET_NAME = 'nguyendv02/ViMD_Dataset'
DRIVE_ROOT = Path('/content/drive/MyDrive/viet_tts_project_data')
DRIVE_OUTPUT_DIR = DRIVE_ROOT / 'vimd_subset_15s_2'
LOCAL_ROOT = Path('/content/viet_tts_project_data')
LOCAL_OUTPUT_DIR = LOCAL_ROOT / 'vimd_subset_15s_2'

HOURS_PER_REGION = 25
MIN_DURATION = 3.0
MAX_DURATION = 17.0
TARGET_SAMPLE_RATE = 24_000
CHECKPOINT_PATH = LOCAL_OUTPUT_DIR / 'manifests' / 'extract_checkpoint.json'
CHECKPOINT_INTERVAL = 200
RESUME_FROM_CHECKPOINT = True

# Keep original short-clip timing for TTS. VAD is used for long-clip chunking only.
USE_VAD_TRIMMING = False
VAD_SAMPLE_RATE = 16_000
VAD_THRESHOLD = 0.5
VAD_MIN_SPEECH_MS = 250
VAD_MIN_SILENCE_MS = 500
VAD_SPEECH_PAD_MS = 120

# Long-clip ASR backend. We use PhoWhisper for Vietnamese ASR.
ASR_BACKEND = 'phowhisper'
PHOWHISPER_MODEL = 'vinai/PhoWhisper-base'
PHOWHISPER_GENERATE_KWARGS = {'language': 'vi', 'task': 'transcribe'}
PHOWHISPER_RETURN_TIMESTAMPS = True
PHOWHISPER_CHUNK_LENGTH_SEC = 15.0
PHOWHISPER_STRIDE_LENGTH_SEC = 0.5

# Fallback: VAD-only long-clip segmentation. This is less accurate, but can still be useful for generating more training data.
SEGMENT_LONG_CLIPS = True
INCLUDE_UNALIGNED_SEGMENTS_IN_TRAINING = False
MAX_SOURCE_DURATION = 30.0
PACK_MAX_GAP_SEC = 0.7
SEGMENT_MIN_DURATION = 3.0
SEGMENT_TARGET_DURATION = 15.0
SEGMENT_HARD_MAX_DURATION = 17.0
SEGMENT_OVERLAP_SEC = 0.25
SEGMENT_PAD_SEC = 0.12

REGIONS = ['North', 'Central', 'South']
SPLITS = ['train']
SPLIT_RATIOS = {
    'train': 1.0,
}

LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Local extraction path: {LOCAL_OUTPUT_DIR}')
print(f'Drive sync path: {DRIVE_OUTPUT_DIR}')
print(f'ASR backend: {ASR_BACKEND}')
print(f'ASR device: {"cuda" if torch.cuda.is_available() else "cpu"}')

Local extraction path: /content/viet_tts_project_data/vimd_subset_15s_2
Drive sync path: /content/drive/MyDrive/viet_tts_project_data/vimd_subset_15s_2
ASR backend: phowhisper
ASR device: cuda


## 5) Helper functions

In [17]:
_vad_model = None
_whisperx_model = None
_phowhisper_processor = None
_phowhisper_model = None


def clean_text(text: str) -> str:
    if text is None:
        return ''

    text = str(text).strip()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\s+([,.!?;:])', r'\1', text)
    return text


def read_audio_array(audio_info: dict):
    source_bytes = audio_info.get('bytes')
    source_path = audio_info.get('path')

    if source_bytes is not None:
        audio_array, sample_rate = sf.read(io.BytesIO(source_bytes))
    elif source_path is not None:
        audio_array, sample_rate = sf.read(source_path)
    else:
        raise ValueError('audio source is missing both bytes and path')

    return audio_array, sample_rate


def to_mono(audio_array):
    if getattr(audio_array, 'ndim', 1) > 1:
        return np.mean(audio_array, axis=1)
    return audio_array


def resample_audio(audio_array, source_sr, target_sr):
    if source_sr == target_sr:
        return audio_array

    y = audio_array.T if getattr(audio_array, 'ndim', 1) > 1 else audio_array
    resampled = librosa.resample(y=y, orig_sr=source_sr, target_sr=target_sr)
    if getattr(resampled, 'ndim', 1) > 1:
        resampled = resampled.T
    return resampled


def get_duration(audio: dict):
    if audio is None:
        return None

    try:
        audio_array, sample_rate = read_audio_array(audio)
        return len(audio_array) / sample_rate
    except Exception as e:
        print(f'[WARN] audio read failed: {e}')
        return None


def is_valid_metadata(row) -> bool:
    text = clean_text(row.get('text', ''))
    region = row.get('region')
    audio = row.get('audio')

    if not text:
        return False
    if region not in REGIONS:
        return False
    if audio is None:
        return False

    return True

def ensure_dirs(output_root: Path):
    (output_root / 'audio').mkdir(parents=True, exist_ok=True)
    (output_root / 'segment_candidates').mkdir(parents=True, exist_ok=True)
    (output_root / 'manifests').mkdir(parents=True, exist_ok=True)

def make_checkpoint_config(dataset_name, hours_per_region, min_dur, max_dur):
    return {
        'dataset_name': dataset_name,
        'hours_per_region': hours_per_region,
        'min_duration': min_dur,
        'max_duration': max_dur,
        'target_sample_rate': TARGET_SAMPLE_RATE,
        'regions': REGIONS,
        'splits': SPLITS,
        'split_ratios': SPLIT_RATIOS,
        'asr_backend': ASR_BACKEND,
        'phowhisper_model': PHOWHISPER_MODEL,
        'phowhisper_generate_kwargs': PHOWHISPER_GENERATE_KWARGS,
        'phowhisper_return_timestamps': PHOWHISPER_RETURN_TIMESTAMPS,
        'phowhisper_chunk_length_sec': PHOWHISPER_CHUNK_LENGTH_SEC,
        'phowhisper_stride_length_sec': PHOWHISPER_STRIDE_LENGTH_SEC,
        'use_vad_trimming': USE_VAD_TRIMMING,
        'vad_sample_rate': VAD_SAMPLE_RATE,
        'vad_threshold': VAD_THRESHOLD,
        'vad_min_speech_ms': VAD_MIN_SPEECH_MS,
        'vad_min_silence_ms': VAD_MIN_SILENCE_MS,
        'vad_speech_pad_ms': VAD_SPEECH_PAD_MS,
        'segment_long_clips': SEGMENT_LONG_CLIPS,
        'include_unaligned_segments_in_training': INCLUDE_UNALIGNED_SEGMENTS_IN_TRAINING,
        'max_source_duration': MAX_SOURCE_DURATION,
        'pack_max_gap_sec': PACK_MAX_GAP_SEC,
        'segment_min_duration': SEGMENT_MIN_DURATION,
        'segment_target_duration': SEGMENT_TARGET_DURATION,
        'segment_hard_max_duration': SEGMENT_HARD_MAX_DURATION,
        'segment_overlap_sec': SEGMENT_OVERLAP_SEC,
        'segment_pad_sec': SEGMENT_PAD_SEC,
    }


def empty_progress():
    return {
        'accepted_sec': {split: {region: 0.0 for region in REGIONS} for split in SPLITS},
        'split_positions': {split: -1 for split in SPLITS},
        'rows': [],
        'segment_candidate_rows': [],
    }


def save_checkpoint(checkpoint_path: Path, config: dict, progress: dict):
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = checkpoint_path.with_suffix(checkpoint_path.suffix + '.tmp')
    payload = {'config': config, **progress}

    with open(tmp_path, 'w', encoding='utf-8') as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    tmp_path.replace(checkpoint_path)


def load_checkpoint(checkpoint_path: Path, config: dict):
    if not checkpoint_path.exists():
        raise FileNotFoundError(f'checkpoint not found: {checkpoint_path}')

    with open(checkpoint_path, 'r', encoding='utf-8') as f:
        checkpoint = json.load(f)

    saved_config = checkpoint.get('config')
    if saved_config != config:
        raise ValueError(
            'checkpoint config does not match current notebook settings. '
            'Use the original settings or set RESUME_FROM_CHECKPOINT=False to start over.'
        )

    progress = empty_progress()
    progress['accepted_sec'].update(checkpoint.get('accepted_sec', {}))
    progress['split_positions'].update(checkpoint.get('split_positions', {}))
    progress['rows'] = checkpoint.get('rows', [])
    progress['segment_candidate_rows'] = checkpoint.get('segment_candidate_rows', [])
    return progress


def get_vad_model():
    global _vad_model
    if _vad_model is None:
        torch.set_num_threads(1)
        _vad_model = load_silero_vad()
    return _vad_model





def get_phowhisper_components():
    global _phowhisper_processor, _phowhisper_model
    if _phowhisper_processor is None:
        _phowhisper_processor = WhisperProcessor.from_pretrained(PHOWHISPER_MODEL)
    if _phowhisper_model is None:
        dtype = torch.float16 if torch.cuda.is_available() else torch.float32
        _phowhisper_model = WhisperForConditionalGeneration.from_pretrained(
            PHOWHISPER_MODEL,
            torch_dtype=dtype,
            use_safetensors=False,
        )
        _phowhisper_model.to(get_phowhisper_device())
        _phowhisper_model.eval()
    return _phowhisper_processor, _phowhisper_model


def get_phowhisper_device():
    return 'cuda' if torch.cuda.is_available() else 'cpu'


def transcribe_with_phowhisper(audio_array, sample_rate):
    mono = to_mono(audio_array).astype(np.float32)
    max_samples = int(SEGMENT_HARD_MAX_DURATION * sample_rate)
    if len(mono) > max_samples:
        mono = mono[:max_samples]

    duration = len(mono) / sample_rate
    if duration > SEGMENT_HARD_MAX_DURATION:
        raise ValueError(
            f'PhoWhisper chunk is {duration:.2f}s, which exceeds '
            f'SEGMENT_HARD_MAX_DURATION={SEGMENT_HARD_MAX_DURATION:.2f}s'
        )

    peak = np.max(np.abs(mono)) if len(mono) else 0.0
    if peak > 1.0:
        mono = mono / peak

    asr_sample_rate = 16_000
    mono_16k = resample_audio(mono, sample_rate, asr_sample_rate).astype(np.float32)
    processor, model = get_phowhisper_components()
    inputs = processor(
        mono_16k,
        sampling_rate=asr_sample_rate,
        return_tensors='pt',
    )

    input_features = inputs.input_features.to(get_phowhisper_device())
    if torch.cuda.is_available():
        input_features = input_features.to(torch.float16)

    generate_kwargs = dict(PHOWHISPER_GENERATE_KWARGS)

    with torch.inference_mode():
        predicted_ids = model.generate(input_features, **generate_kwargs)

    transcript = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    return clean_text(transcript)


def split_segment_spans(segments, source_duration):
    clipped_segments = []
    max_duration = min(float(SEGMENT_HARD_MAX_DURATION), 29.0)
    stride = max_duration - float(SEGMENT_OVERLAP_SEC)
    if stride <= 0:
        raise ValueError('SEGMENT_OVERLAP_SEC must be smaller than SEGMENT_HARD_MAX_DURATION')

    for start_sec, end_sec, interval_count in segments:
        start_sec = max(0.0, float(start_sec) - SEGMENT_PAD_SEC)
        end_sec = min(float(end_sec) + SEGMENT_PAD_SEC, source_duration)
        if end_sec <= start_sec:
            continue

        cursor = start_sec
        while cursor < end_sec:
            chunk_end = min(cursor + max_duration, end_sec)
            duration = chunk_end - cursor

            if duration < SEGMENT_MIN_DURATION:
                if clipped_segments and interval_count == clipped_segments[-1][2]:
                    prev_start, prev_end, prev_count = clipped_segments[-1]
                    merged_end = min(end_sec, prev_end + duration)
                    if merged_end - prev_start <= max_duration:
                        clipped_segments[-1] = (prev_start, merged_end, prev_count)
                break

            clipped_segments.append((cursor, chunk_end, interval_count))
            if chunk_end >= end_sec:
                break
            cursor += stride

    return clipped_segments


def slice_audio_segment(audio_array, sample_rate, start_sec, end_sec):
    total_samples = len(audio_array)
    max_samples = int(SEGMENT_HARD_MAX_DURATION * sample_rate)
    start_sample = max(0, min(int(round(start_sec * sample_rate)), total_samples))
    requested_end_sample = max(start_sample, min(int(round(end_sec * sample_rate)), total_samples))
    end_sample = min(requested_end_sample, start_sample + max_samples, total_samples)

    segment_audio = audio_array[start_sample:end_sample]
    segment_duration = len(segment_audio) / sample_rate
    clipped_start_sec = start_sample / sample_rate
    clipped_end_sec = end_sample / sample_rate
    return segment_audio, segment_duration, clipped_start_sec, clipped_end_sec








def detect_speech_intervals(audio_array, sample_rate):
    mono = to_mono(audio_array).astype(np.float32)
    vad_audio = resample_audio(mono, sample_rate, VAD_SAMPLE_RATE)
    vad_audio = np.asarray(vad_audio, dtype=np.float32)

    peak = np.max(np.abs(vad_audio)) if len(vad_audio) else 0.0
    if peak > 1.0:
        vad_audio = vad_audio / peak

    timestamps = get_speech_timestamps(
        torch.from_numpy(vad_audio),
        get_vad_model(),
        sampling_rate=VAD_SAMPLE_RATE,
        threshold=VAD_THRESHOLD,
        min_speech_duration_ms=VAD_MIN_SPEECH_MS,
        min_silence_duration_ms=VAD_MIN_SILENCE_MS,
        speech_pad_ms=VAD_SPEECH_PAD_MS,
        max_speech_duration_s=SEGMENT_HARD_MAX_DURATION,
        return_seconds=True,
    )
    return [(float(ts['start']), float(ts['end'])) for ts in timestamps]


def trim_to_speech_bounds(audio_array, sample_rate):
    intervals = detect_speech_intervals(audio_array, sample_rate)
    if not intervals:
        return audio_array, 0.0, len(audio_array) / sample_rate, False

    start_sec = max(0.0, intervals[0][0])
    end_sec = min(len(audio_array) / sample_rate, intervals[-1][1])
    start_sample = int(start_sec * sample_rate)
    end_sample = int(end_sec * sample_rate)
    return audio_array[start_sample:end_sample], start_sec, end_sec, True


def pack_speech_intervals(intervals):
    packs = []
    current = []

    for start, end in intervals:
        if not current:
            current = [[start, end]]
            continue

        gap = start - current[-1][1]
        current_duration = current[-1][1] - current[0][0]
        packed_duration = end - current[0][0]
        should_start_new = (
            gap > PACK_MAX_GAP_SEC
            or packed_duration > SEGMENT_HARD_MAX_DURATION
            or current_duration >= SEGMENT_TARGET_DURATION
        )

        if should_start_new:
            packs.append(current)
            current = [[start, end]]
        else:
            current.append([start, end])

    if current:
        packs.append(current)

    segments = []
    for pack in packs:
        start = pack[0][0]
        end = pack[-1][1]
        duration = end - start
        if SEGMENT_MIN_DURATION <= duration <= SEGMENT_HARD_MAX_DURATION:
            segments.append((start, end, len(pack)))

    return segments





def save_audio_array(audio_array, sample_rate, output_path: Path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    audio_array = resample_audio(audio_array, sample_rate, TARGET_SAMPLE_RATE)
    sf.write(output_path, audio_array, TARGET_SAMPLE_RATE)


def save_whole_or_trimmed_audio(audio_info: dict, output_path: Path):
    audio_array, sample_rate = read_audio_array(audio_info)
    original_duration = len(audio_array) / sample_rate

    trim_start_sec = 0.0
    trim_end_sec = original_duration
    used_vad = False

    if USE_VAD_TRIMMING:
        trimmed_audio, trim_start_sec, trim_end_sec, used_vad = trim_to_speech_bounds(audio_array, sample_rate)
        trimmed_duration = len(trimmed_audio) / sample_rate
        if trimmed_duration >= MIN_DURATION:
            audio_array = trimmed_audio

    save_audio_array(audio_array, sample_rate, output_path)
    saved_duration = len(audio_array) / sample_rate
    return saved_duration, trim_start_sec, trim_end_sec, used_vad





def build_phowhisper_segment_rows(row, split_name, idx, output_root):
    audio_array, sample_rate = read_audio_array(row['audio'])
    source_duration = len(audio_array) / sample_rate
    intervals = detect_speech_intervals(audio_array, sample_rate)
    segments = pack_speech_intervals(intervals)

    if not segments and source_duration >= SEGMENT_MIN_DURATION:
        segments = [(0.0, min(source_duration, SEGMENT_HARD_MAX_DURATION), 1)]
    segments = split_segment_spans(segments, source_duration)

    region = row['region']
    filename = row.get('filename', f'{split_name}_{idx}')
    base = Path(filename).stem
    rows = []

    for seg_idx, (start_sec, end_sec, interval_count) in enumerate(segments):
        segment_audio, segment_duration, clipped_start_sec, clipped_end_sec = slice_audio_segment(
            audio_array,
            sample_rate,
            start_sec,
            end_sec,
        )
        if not (SEGMENT_MIN_DURATION <= segment_duration <= SEGMENT_HARD_MAX_DURATION):
            print(f'[WARN] skipping {filename} segment {seg_idx}: duration={segment_duration:.2f}s')
            continue

        try:
            transcript = transcribe_with_phowhisper(segment_audio, sample_rate)
        except Exception as e:
            print(f'[WARN] PhoWhisper failed for {filename} segment {seg_idx} ({segment_duration:.2f}s): {e}')
            continue

        if not transcript:
            continue

        wav_path = output_root / 'audio' / split_name / region / f'{split_name}_{base}_pw{seg_idx:03d}.wav'
        save_audio_array(segment_audio, sample_rate, wav_path)

        rows.append({
            'audio_path': str(wav_path.resolve()),
            'text': transcript,
            'source_text': clean_text(row['text']),
            'duration_sec': round(segment_duration, 3),
            'original_duration_sec': round(source_duration, 3),
            'region': region,
            'split': split_name,
            'filename': filename,
            'source_start_sec': round(clipped_start_sec, 3),
            'source_end_sec': round(clipped_end_sec, 3),
            'requested_start_sec': round(start_sec, 3),
            'requested_end_sec': round(end_sec, 3),
            'vad_interval_count': interval_count,
            'alignment_method': 'phowhisper_vad_chunk',
            'phowhisper_model': PHOWHISPER_MODEL,
            'needs_text_alignment': False,
        })

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return rows


def build_long_clip_segment_candidates(row, split_name, idx, output_root):
    audio_array, sample_rate = read_audio_array(row['audio'])
    intervals = detect_speech_intervals(audio_array, sample_rate)
    segments = pack_speech_intervals(intervals)

    text = clean_text(row['text'])
    region = row['region']
    filename = row.get('filename', f'{split_name}_{idx}')
    base = Path(filename).stem
    rows = []

    for seg_idx, (start_sec, end_sec, interval_count) in enumerate(segments):
        start_sample = int(start_sec * sample_rate)
        end_sample = int(end_sec * sample_rate)
        segment_audio = audio_array[start_sample:end_sample]
        segment_duration = len(segment_audio) / sample_rate

        wav_path = (
            output_root
            / 'segment_candidates'
            / split_name
            / region
            / f'{split_name}_{base}_seg{seg_idx:03d}.wav'
        )
        save_audio_array(segment_audio, sample_rate, wav_path)

        rows.append({
            'audio_path': str(wav_path.resolve()),
            'text': text if INCLUDE_UNALIGNED_SEGMENTS_IN_TRAINING else '',
            'source_text': text,
            'duration_sec': round(segment_duration, 3),
            'region': region,
            'split': split_name,
            'filename': filename,
            'source_start_sec': round(start_sec, 3),
            'source_end_sec': round(end_sec, 3),
            'vad_interval_count': interval_count,
            'alignment_method': 'silero_vad_only',
            'needs_text_alignment': True,
        })

    return rows


def build_targets(hours_per_region: float):
    total_sec = hours_per_region * 3600
    return {
        split: {region: total_sec * SPLIT_RATIOS[split] for region in REGIONS}
        for split in SPLITS
    }


def build_manifest_rows(rows):
    return [
        {
            'audio_file': row['audio_path'],
            'text': f"[{row['region']}] {row['text']}",
        }
        for row in rows
        if row.get('text') and not row.get('needs_text_alignment', False)
    ]

## 6) Preview a few streamed rows

In [18]:
sample_stream = load_dataset(DATASET_NAME, split='train', streaming=True)
sample_stream = sample_stream.decode(False).with_format('python')

preview_rows = []
for i, row in enumerate(sample_stream):
    duration = get_duration(row.get('audio'))
    preview_rows.append({
        'filename': row.get('filename'),
        'region': row.get('region'),
        'text': row.get('text'),
        'duration_sec': duration,
        'passes_metadata_filter': is_valid_metadata(row),
        'is_short_clip': duration is not None and MIN_DURATION <= duration <= MAX_DURATION,
        'will_try_phowhisper_chunking': duration is not None and duration > MAX_DURATION,
    })
    if i >= 9:
        break

pd.DataFrame(preview_rows)

Resolving data files:   0%|          | 0/103 [00:00<?, ?it/s]

,filename,region,text,duration_sec,passes_metadata_filter,is_short_clip,will_try_phowhisper_chunking
0,11_0001.wav,North,"Nghiên cứu học tập,các ứng dụng các khoa học c...",24.243991,True,False,True
1,11_0002.wav,North,Chuyển đổi số hiện nay đang được là đang được ...,21.279002,True,False,True
2,11_0003.wav,North,Trong quá trình mình làm việc để quản lý thông...,26.920998,True,False,True
3,11_0004.wav,North,Ý tưởng đầu tiên tôi phải nhắc đến là vấn đề v...,22.505011,True,False,True
4,11_0005.wav,North,giúp giải quyết đa dạng các nội dung trong quả...,22.458005,True,False,True
5,11_0006.wav,North,Em rất là tự hào vì năm đầu tiên em được tham ...,15.179705,True,True,False
6,11_0007.wav,North,"Em cảm thấy rất là tự hào và qua cái lễ hội, l...",30.537596,True,False,True
7,11_0008.wav,North,Năm nay thì Cao Trương tôi thì cũng chuẩn bị n...,29.761270,True,False,True
8,11_0009.wav,North,Em thưa chị là em cảm thấy rất là vinh dự và t...,29.861791,True,False,True
9,11_0010.wav,North,"Qua thực tế, thì cán bộ công nhân viên từ kỹ t...",19.541519,True,False,True


## 7) Extraction function

In [19]:
def extract_subset(
    dataset_name,
    output_root,
    hours_per_region,
    min_dur,
    max_dur,
    checkpoint_path=None,
    checkpoint_interval=200,
    resume=False,
):
    ensure_dirs(output_root)
    targets = build_targets(hours_per_region)
    config = make_checkpoint_config(dataset_name, hours_per_region, min_dur, max_dur)

    if checkpoint_path is None:
        checkpoint_path = output_root / 'manifests' / 'extract_checkpoint.json'
    if checkpoint_interval < 1:
        raise ValueError('checkpoint_interval must be at least 1')

    if resume:
        progress = load_checkpoint(checkpoint_path, config)
        print(f'[INFO] Resuming from checkpoint: {checkpoint_path}')
    else:
        progress = empty_progress()

    accepted_sec = progress['accepted_sec']
    split_positions = progress['split_positions']
    rows = progress['rows']
    segment_candidate_rows = progress['segment_candidate_rows']
    last_checkpoint_total = len(rows) + len(segment_candidate_rows)

    print('\n[INFO] Targets:')
    for split in SPLITS:
        for region in REGIONS:
            print(f"  {split} | {region}: {targets[split][region] / 3600:.2f}h")

    try:
        for split_name in SPLITS:
            if all(accepted_sec[split_name][r] >= targets[split_name][r] for r in REGIONS):
                print(f'\n[INFO] Skipping completed split from checkpoint: {split_name}')
                continue

            print(f'\n[INFO] Streaming split: {split_name}')
            ds_stream = load_dataset(dataset_name, split=split_name, streaming=True)
            ds_stream = ds_stream.decode(False).with_format('python')

            for idx, row in enumerate(ds_stream):
                if idx <= split_positions[split_name]:
                    continue

                split_positions[split_name] = idx

                if all(accepted_sec[split_name][r] >= targets[split_name][r] for r in REGIONS):
                    print(f'[INFO] Finished split: {split_name}')
                    save_checkpoint(checkpoint_path, config, progress)
                    break

                if not is_valid_metadata(row):
                    continue

                region = row['region']
                audio = row['audio']

                if accepted_sec[split_name][region] >= targets[split_name][region]:
                    continue

                duration = get_duration(audio)
                if duration is None or duration < min_dur:
                    continue
                if MAX_SOURCE_DURATION is not None and duration > MAX_SOURCE_DURATION:
                    continue

                text = clean_text(row['text'])
                filename = row.get('filename', f'{split_name}_{idx}')
                base = Path(filename).stem

                if duration <= max_dur:
                    wav_path = output_root / 'audio' / split_name / region / f'{split_name}_{base}.wav'

                    try:
                        saved_duration, trim_start_sec, trim_end_sec, used_vad = save_whole_or_trimmed_audio(audio, wav_path)
                    except Exception as e:
                        print(f'[WARN] failed to save {filename}: {e}')
                        continue

                    if not (min_dur <= saved_duration <= max_dur):
                        continue

                    rows.append({
                        'audio_path': str(wav_path.resolve()),
                        'text': text,
                        'duration_sec': round(saved_duration, 3),
                        'original_duration_sec': round(duration, 3),
                        'region': region,
                        'split': split_name,
                        'filename': filename,
                        'vad_trimmed': used_vad,
                        'trim_start_sec': round(trim_start_sec, 3),
                        'trim_end_sec': round(trim_end_sec, 3),
                        'alignment_method': 'original_transcript',
                        'needs_text_alignment': False,
                    })

                    accepted_sec[split_name][region] += saved_duration

                elif SEGMENT_LONG_CLIPS:
                    aligned_rows = []
                    if ASR_BACKEND == 'phowhisper':
                        try:
                            aligned_rows = build_phowhisper_segment_rows(row, split_name, idx, output_root)
                        except Exception as e:
                            print(f'[WARN] PhoWhisper failed for {filename}; falling back to VAD candidates: {e}')
                    else:
                        print(f'[WARN] ASR backend {ASR_BACKEND!r} is not enabled for {filename}; falling back to VAD candidates')

                    if aligned_rows and ASR_BACKEND == 'phowhisper':
                        for aligned_row in aligned_rows:
                            if accepted_sec[split_name][region] >= targets[split_name][region]:
                                break
                            rows.append(aligned_row)
                            accepted_sec[split_name][region] += aligned_row['duration_sec']
                    else:
                        try:
                            candidates = build_long_clip_segment_candidates(row, split_name, idx, output_root)
                        except Exception as e:
                            print(f'[WARN] failed to segment {filename}: {e}')
                            continue

                        segment_candidate_rows.extend(candidates)

                        if INCLUDE_UNALIGNED_SEGMENTS_IN_TRAINING:
                            for candidate in candidates:
                                if accepted_sec[split_name][region] >= targets[split_name][region]:
                                    break
                                rows.append(candidate)
                                accepted_sec[split_name][region] += candidate['duration_sec']

                total_saved_items = len(rows) + len(segment_candidate_rows)
                if total_saved_items - last_checkpoint_total >= checkpoint_interval:
                    print(
                        f'[INFO] Checkpointing {len(rows)} training samples and '
                        f'{len(segment_candidate_rows)} VAD-only candidates...'
                    )
                    save_checkpoint(checkpoint_path, config, progress)
                    last_checkpoint_total = total_saved_items

            save_checkpoint(checkpoint_path, config, progress)
    except KeyboardInterrupt:
        save_checkpoint(checkpoint_path, config, progress)
        print(f'\n[INFO] Interrupted. Checkpoint saved to: {checkpoint_path}')
        raise

    return rows, accepted_sec, segment_candidate_rows

## 8) Run extraction

This may take a while for the full 20 hours per region.
During extraction, files are written to fast local Colab storage.

Long clips now use `vinai/PhoWhisper-base`. Silero VAD creates 15s-ish chunks and PhoWhisper transcribes each chunk.

Checkpoints are written to `CHECKPOINT_PATH` during extraction. Set `RESUME_FROM_CHECKPOINT=True` in the config cell before rerunning this step to continue from a saved checkpoint.

In [20]:
print("LOCAL_OUTPUT_DIR:", LOCAL_OUTPUT_DIR)
print("DRIVE_OUTPUT_DIR:", DRIVE_OUTPUT_DIR)
print("CHECKPOINT_PATH:", CHECKPOINT_PATH)

LOCAL_OUTPUT_DIR: /content/viet_tts_project_data/vimd_subset_15s_2
DRIVE_OUTPUT_DIR: /content/drive/MyDrive/viet_tts_project_data/vimd_subset_15s_2
CHECKPOINT_PATH: /content/viet_tts_project_data/vimd_subset_15s_2/manifests/extract_checkpoint.json


In [21]:
rows, accepted_sec, segment_candidate_rows = extract_subset(
    dataset_name=DATASET_NAME,
    output_root=LOCAL_OUTPUT_DIR,
    hours_per_region=HOURS_PER_REGION,
    min_dur=MIN_DURATION,
    max_dur=MAX_DURATION,
    checkpoint_path=CHECKPOINT_PATH,
    checkpoint_interval=CHECKPOINT_INTERVAL,
    resume=RESUME_FROM_CHECKPOINT,
)

print(f'Selected training rows: {len(rows)}')
print(f'VAD-only candidates needing alignment: {len(segment_candidate_rows)}')

[INFO] Resuming from checkpoint: /content/viet_tts_project_data/vimd_subset_15s_2/manifests/extract_checkpoint.json

[INFO] Targets:
  train | North: 25.00h
  train | Central: 25.00h
  train | South: 25.00h

[INFO] Streaming split: train


Resolving data files:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/246 [00:00<?, ?it/s]

[INFO] Checkpointing 16639 training samples and 0 VAD-only candidates...
[INFO] Checkpointing 16840 training samples and 0 VAD-only candidates...
[INFO] Checkpointing 17040 training samples and 0 VAD-only candidates...
[INFO] Checkpointing 17240 training samples and 0 VAD-only candidates...
[INFO] Checkpointing 17441 training samples and 0 VAD-only candidates...
[INFO] Checkpointing 17641 training samples and 0 VAD-only candidates...
[INFO] Checkpointing 17842 training samples and 0 VAD-only candidates...
[INFO] Checkpointing 18042 training samples and 0 VAD-only candidates...
[INFO] Checkpointing 18242 training samples and 0 VAD-only candidates...
[INFO] Checkpointing 18443 training samples and 0 VAD-only candidates...
[INFO] Checkpointing 18644 training samples and 0 VAD-only candidates...
[INFO] Checkpointing 18845 training samples and 0 VAD-only candidates...
[INFO] Checkpointing 19045 training samples and 0 VAD-only candidates...
[INFO] Checkpointing 19245 training samples and 0 V

In [22]:
from pathlib import Path
import shutil

src = Path('/content/viet_tts_project_data/vimd_subset_15s_2/manifests/extract_checkpoint.json')
dst = Path('/content/drive/MyDrive/viet_tts_project_data/vimd_subset_15s_2/manifests/extract_checkpoint.json')

dst.parent.mkdir(parents=True, exist_ok=True)
if src.exists():
    shutil.copy2(src, dst)
    print(f'Copied checkpoint to: {dst}')
else:
    print(f'Source checkpoint does not exist: {src}')
print(f'Exists on Drive: {dst.exists()}')

Copied checkpoint to: /content/drive/MyDrive/viet_tts_project_data/vimd_subset_15s_2/manifests/extract_checkpoint.json
Exists on Drive: True


In [23]:
from pathlib import Path
import shutil

# Paths
local_root = Path('/content/viet_tts_project_data/vimd_subset_15s_2')
drive_root = Path('/content/drive/MyDrive/viet_tts_project_data/vimd_subset_15s_2')

# Copy train audio
src_audio = local_root / 'audio' / 'train'
dst_audio = drive_root / 'audio' / 'train'

if src_audio.exists():
    shutil.copytree(src_audio, dst_audio, dirs_exist_ok=True)
    print('Train audio copied')

# Copy manifests
src_manifest = local_root / 'manifests'
dst_manifest = drive_root / 'manifests'

if src_manifest.exists():
    shutil.copytree(src_manifest, dst_manifest, dirs_exist_ok=True)
    print('Manifests copied')

Train audio copied
Manifests copied


In [32]:
from pathlib import Path

local_train = Path('/content/viet_tts_project_data/vimd_subset_15s_2/audio/train')
drive_train = Path('/content/drive/MyDrive/viet_tts_project_data/vimd_subset_15s_2/audio/train')

for region in ["North", "Central", "South"]:
    local_count = len(list((local_train / region).glob("*.wav")))
    drive_count = len(list((drive_train / region).glob("*.wav")))
    print(region, "local:", local_count, "| drive:", drive_count)

North local: 8008 | drive: 8008
Central local: 7615 | drive: 7615
South local: 7206 | drive: 7206


In [25]:
# from pathlib import Path

# drive_train = Path('/content/drive/MyDrive/viet_tts_project_data/vimd_subset_15s_2/audio/train')

# print("Folders in Drive train:")
# for p in drive_train.iterdir():
#     print(p.name, "is_dir:", p.is_dir())

# print("\nCounts:")
# for region in ["North", "Central", "South"]:
#     folder = drive_train / region
#     print(region, folder.exists(), len(list(folder.glob("*.wav"))))

In [26]:
# from pathlib import Path
# import shutil

# local_south = Path('/content/viet_tts_project_data/vimd_subset_15s_2/audio/train/South')
# drive_south = Path('/content/drive/MyDrive/viet_tts_project_data/vimd_subset_15s_2/audio/train/South')

# drive_south.mkdir(parents=True, exist_ok=True)

# copied = 0
# skipped = 0

# for wav in local_south.glob("*.wav"):
#     dst = drive_south / wav.name
#     if not dst.exists():
#         shutil.copy2(wav, dst)
#         copied += 1
#     else:
#         skipped += 1

# print(f"Copied {copied} missing files, skipped {skipped}")

In [27]:
# from pathlib import Path

# drive_train = Path('/content/drive/MyDrive/viet_tts_project_data/vimd_subset_15s_2/audio/train')

# print("Folders in Drive train:")
# for p in drive_train.iterdir():
#     print(p.name, "is_dir:", p.is_dir())

# print("\nCounts:")
# for region in ["North", "Central", "South"]:
#     folder = drive_train / region
#     print(region, folder.exists(), len(list(folder.glob("*.wav"))))

## 9) Inspect results

In [28]:
df_rows = pd.DataFrame(rows)
df_segment_candidates = pd.DataFrame(segment_candidate_rows)

display(df_rows.head(8))
if len(df_segment_candidates) > 0:
    display(df_segment_candidates.head(8))


,audio_path,text,source_text,duration_sec,original_duration_sec,region,split,filename,source_start_sec,source_end_sec,requested_start_sec,requested_end_sec,vad_interval_count,alignment_method,phowhisper_model,needs_text_alignment,vad_trimmed,trim_start_sec,trim_end_sec
0,/content/viet_tts_project_data/vimd_subset_15s...,nghiên cứu học tập các ứng dụng các khoa công ...,"Nghiên cứu học tập,các ứng dụng các khoa học c...",8.320,24.244,North,train,11_0001.wav,0.00,8.320,0.00,8.320,1.0,phowhisper_vad_chunk,vinai/PhoWhisper-base,False,NaN,NaN,NaN
1,/content/viet_tts_project_data/vimd_subset_15s...,trong công việc của mình từ đó rất chuyển đổi ...,"Nghiên cứu học tập,các ứng dụng các khoa học c...",12.940,24.244,North,train,11_0001.wav,8.18,21.120,8.18,21.120,2.0,phowhisper_vad_chunk,vinai/PhoWhisper-base,False,NaN,NaN,NaN
2,/content/viet_tts_project_data/vimd_subset_15s...,chuyển hồ số hiện nay đang được đảng và nhận r...,Chuyển đổi số hiện nay đang được là đang được ...,10.040,21.279,North,train,11_0002.wav,0.08,10.120,0.08,10.120,1.0,phowhisper_vad_chunk,vinai/PhoWhisper-base,False,NaN,NaN,NaN
3,/content/viet_tts_project_data/vimd_subset_15s...,thực hiện những cái công trình chuyển đổi số đ...,Chuyển đổi số hiện nay đang được là đang được ...,11.299,21.279,North,train,11_0002.wav,9.98,21.279,9.98,21.279,1.0,phowhisper_vad_chunk,vinai/PhoWhisper-base,False,NaN,NaN,NaN
4,/content/viet_tts_project_data/vimd_subset_15s...,trong quá trình mình làm việc để quản lý.,Trong quá trình mình làm việc để quản lý thông...,3.620,26.921,North,train,11_0003.wav,0.00,3.620,0.00,3.620,1.0,phowhisper_vad_chunk,vinai/PhoWhisper-base,False,NaN,NaN,NaN
5,/content/viet_tts_project_data/vimd_subset_15s...,thông tin các vật tư thiết bị của công ty điện...,Trong quá trình mình làm việc để quản lý thông...,17.000,26.921,North,train,11_0003.wav,3.68,20.680,3.68,20.680,1.0,phowhisper_vad_chunk,vinai/PhoWhisper-base,False,NaN,NaN,NaN
6,/content/viet_tts_project_data/vimd_subset_15s...,kiểm kê nhanh chóng chính xác an toàn nên tôi ...,Trong quá trình mình làm việc để quản lý thông...,6.241,26.921,North,train,11_0003.wav,20.68,26.921,20.68,26.921,1.0,phowhisper_vad_chunk,vinai/PhoWhisper-base,False,NaN,NaN,NaN
7,/content/viet_tts_project_data/vimd_subset_15s...,ý tưởng đầu tiên tôi phải nhắc đến là vấn đề c...,Ý tưởng đầu tiên tôi phải nhắc đến là vấn đề v...,7.140,22.505,North,train,11_0004.wav,0.08,7.220,0.08,7.220,1.0,phowhisper_vad_chunk,vinai/PhoWhisper-base,False,NaN,NaN,NaN


In [29]:
if len(rows) > 0:
    display(df_rows.groupby(['split', 'region']).agg(
        samples=('filename', 'count'),
        total_seconds=('duration_sec', 'sum'),
        avg_seconds=('duration_sec', 'mean'),
    ).reset_index())

    display(df_rows.groupby(['alignment_method']).agg(
        samples=('filename', 'count'),
        total_seconds=('duration_sec', 'sum'),
        avg_seconds=('duration_sec', 'mean'),
    ).reset_index())
else:
    print('No training rows selected yet.')

if len(df_segment_candidates) > 0:
    display(df_segment_candidates.groupby(['split', 'region']).agg(
        candidate_segments=('filename', 'count'),
        candidate_seconds=('duration_sec', 'sum'),
        avg_seconds=('duration_sec', 'mean'),
    ).reset_index())
else:
    print('No VAD-only segment candidates were written.')


,split,region,samples,total_seconds,avg_seconds
0,train,Central,7615,85336.769,11.206404
1,train,North,8007,90003.343,11.240582
2,train,South,7206,79994.500,11.101096


,alignment_method,samples,total_seconds,avg_seconds
0,original_transcript,4935,63488.224,12.864888
1,phowhisper_vad_chunk,17893,191846.388,10.721868


No VAD-only segment candidates were written.


## 10) Save manifests locally

In [30]:
manifest_dir = LOCAL_OUTPUT_DIR / 'manifests'
manifest_dir.mkdir(parents=True, exist_ok=True)

if len(rows) == 0:
    print('No training samples were selected, so no training manifest files were written.')
else:
    manifest_rows = build_manifest_rows(rows)

    row_fieldnames = sorted({key for row in rows for key in row.keys()})
    with open(manifest_dir / 'manifest_original.csv', 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=row_fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    with open(manifest_dir / 'manifest.csv', 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(
            f,
            fieldnames=['audio_file', 'text'],
            delimiter='|',
        )
        writer.writeheader()
        writer.writerows(manifest_rows)

    with open(manifest_dir / 'manifest.jsonl', 'w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')

    print(f'Saved {len(rows)} training samples.')
    print(f'Original manifest CSV: {manifest_dir / "manifest_original.csv"}')
    print(f'Training manifest CSV: {manifest_dir / "manifest.csv"}')
    print(f'Manifest JSONL: {manifest_dir / "manifest.jsonl"}')
    print(f'Local audio root: {LOCAL_OUTPUT_DIR / "audio"}')

if len(segment_candidate_rows) > 0:
    candidate_manifest = manifest_dir / 'vad_candidates_needing_alignment.jsonl'
    candidate_csv = manifest_dir / 'vad_candidates_needing_alignment.csv'

    with open(candidate_manifest, 'w', encoding='utf-8') as f:
        for row in segment_candidate_rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')

    candidate_fieldnames = sorted({key for row in segment_candidate_rows for key in row.keys()})
    with open(candidate_csv, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=candidate_fieldnames)
        writer.writeheader()
        writer.writerows(segment_candidate_rows)

    print(f'VAD-only candidate JSONL: {candidate_manifest}')
    print(f'VAD-only candidate CSV: {candidate_csv}')
    print(f'VAD-only candidate audio root: {LOCAL_OUTPUT_DIR / "segment_candidates"}')


Saved 22828 training samples.
Original manifest CSV: /content/viet_tts_project_data/vimd_subset_15s_2/manifests/manifest_original.csv
Training manifest CSV: /content/viet_tts_project_data/vimd_subset_15s_2/manifests/manifest.csv
Manifest JSONL: /content/viet_tts_project_data/vimd_subset_15s_2/manifests/manifest.jsonl
Local audio root: /content/viet_tts_project_data/vimd_subset_15s_2/audio


## 11) Sync finished dataset to Drive

Run this after extraction completes. Copying one finished folder is usually much faster than writing every file straight to Drive.

In [31]:
if not LOCAL_OUTPUT_DIR.exists():
    print(f'Local output folder does not exist: {LOCAL_OUTPUT_DIR}')
else:
    if DRIVE_OUTPUT_DIR.exists():
        shutil.rmtree(DRIVE_OUTPUT_DIR)

    shutil.copytree(LOCAL_OUTPUT_DIR, DRIVE_OUTPUT_DIR)
    print(f'Synced dataset to Drive: {DRIVE_OUTPUT_DIR}')
    print(f'Drive original manifest: {DRIVE_OUTPUT_DIR / "manifests" / "manifest_original.csv"}')
    print(f'Drive training manifest: {DRIVE_OUTPUT_DIR / "manifests" / "manifest.csv"}')

Synced dataset to Drive: /content/drive/MyDrive/viet_tts_project_data/vimd_subset_15s_2
Drive original manifest: /content/drive/MyDrive/viet_tts_project_data/vimd_subset_15s_2/manifests/manifest_original.csv
Drive training manifest: /content/drive/MyDrive/viet_tts_project_data/vimd_subset_15s_2/manifests/manifest.csv


In [1]:
!zip -r /content/vimd_subset_15s_2.zip /content/viet_tts_project_data/vimd_subset_15s_2

updating: content/viet_tts_project_data/vimd_subset_15s_2/ (stored 0%)
updating: content/viet_tts_project_data/vimd_subset_15s_2/audio/ (stored 0%)
updating: content/viet_tts_project_data/vimd_subset_15s_2/audio/train/ (stored 0%)
updating: content/viet_tts_project_data/vimd_subset_15s_2/audio/train/North/ (stored 0%)
updating: content/viet_tts_project_data/vimd_subset_15s_2/audio/train/North/train_14_0193_pw001.wav (deflated 22%)
updating: content/viet_tts_project_data/vimd_subset_15s_2/audio/train/North/train_34_0052_pw000.wav (deflated 43%)
updating: content/viet_tts_project_data/vimd_subset_15s_2/audio/train/North/train_21_0146_pw000.wav (deflated 26%)
updating: content/viet_tts_project_data/vimd_subset_15s_2/audio/train/North/train_27_0169_pw000.wav (deflated 42%)
updating: content/viet_tts_project_data/vimd_subset_15s_2/audio/train/North/train_14_0005_pw000.wav (deflated 33%)
updating: content/viet_tts_project_data/vimd_subset_15s_2/audio/train/North/train_14_0019_pw002.wav (defl

In [2]:
from pathlib import Path

zip_path = Path("/content/vimd_subset_15s_2.zip")
print("Zip exists:", zip_path.exists(), "Size:", zip_path.stat().st_size)

Zip exists: True Size: 15332633269


In [3]:
import shutil
from pathlib import Path

drive_root = Path('/content/drive/MyDrive/viet_tts_project_data/vimd_subset_15s_2')

if drive_root.exists():
    shutil.rmtree(drive_root)
    print("Old dataset deleted from Drive")

Old dataset deleted from Drive


In [5]:
from pathlib import Path
import shutil

src = Path("/content/vimd_subset_15s_2.zip")
dst = Path("/content/drive/MyDrive/viet_tts_project_data/vimd_subset_15s_2.zip")

# create parent folder
dst.parent.mkdir(parents=True, exist_ok=True)

# copy file
shutil.copy2(src, dst)

print("Copied:", dst.exists())
print("Size:", dst.stat().st_size if dst.exists() else "N/A")

Copied: True
Size: 15332633269


In [8]:
from pathlib import Path

zip_local = Path("/content/vimd_subset_15s_2.zip")
zip_drive = Path("/content/drive/MyDrive/viet_tts_project_data/vimd_subset_15s_2.zip")

print("Local zip exists:", zip_local.exists())
print("Drive zip exists:", zip_drive.exists())

Local zip exists: True
Drive zip exists: True


In [9]:
from pathlib import Path

p = Path("/content/drive/MyDrive/viet_tts_project_data/vimd_subset_15s_2.zip")

print("Exists:", p.exists())
if p.exists():
    print("Size:", p.stat().st_size)

Exists: True
Size: 15332633269
